# Homework 08
This homework is based on the clustering lectures. Check the lecture notes and TA notes - they should help!

## Question 1
This question will walk you through creating your own `kmeans` function.

#### a) What are the steps of `kmeans`?
**Hint**: There are 4 steps/builder functions that you'll need.

In [ ]:
- Assign points to cluster at random 
- Compute Cluster Means 
- Reassign points to the nearest centroid 
- Recompute centroids and repeat until convergence.

#### b) Create the builder function for step 1.

In [2]:
random_assign <- function(df, k = 3, prefix = "C", seed = NULL) {
  if (!is.null(seed)) set.seed(seed)
  names <- paste0(prefix, 0:(k-1))
  df %>% mutate(cluster = sample(names, n(), replace = TRUE))
}

#### c) Create the builder function for step 2.

In [3]:
compute_centroids <- function(df, x = "x", y = "y", cluster_col = "cluster") {
  x_s <- sym(x); y_s <- sym(y); cl_s <- sym(cluster_col)
  df %>%
    group_by(!!cl_s) %>%
    summarise(x = mean(!!x_s), y = mean(!!y_s), .groups = "drop") %>%
    arrange(!!cl_s) %>%
    mutate(label = paste0("μ", row_number() - 1))
}

#### d) Create the builder function for step 3.
*Hint*: There are two ways to do this part - one is significantly more efficient than the other. You can do either.  

In [4]:
assign_to_nearest <- function(df_points, centers, x = "x", y = "y", cluster_col = "cluster") {
  cx <- sym(x); cy <- sym(y)
  centers_local <- centers %>% select(cluster = 1, !!cx, !!cy)
  df_points %>%
    rowwise() %>%
    mutate(
      !!sym(cluster_col) := centers_local$cluster[which.min(sqrt((!!cx - centers_local[[2]])^2 + (!!cy - centers_local[[3]])^2))]
    ) %>%
    ungroup()
}

#### e) Create the builder function for step 4.

In [5]:
centers_moved <- function(old, new, tol = 1e-6) {
  oc <- old %>% arrange(cluster) %>% select(x, y) %>% as.matrix()
  nc <- new %>% arrange(cluster) %>% select(x, y) %>% as.matrix()
  if (!all(dim(oc) == dim(nc))) return(TRUE)    # treat as moved if shapes differ
  max(sqrt(rowSums((oc - nc)^2))) > tol
}


#### f) Combine them all into your own `kmeans` function.

In [11]:
library(dplyr)
library(rlang)

kmeans_tidy <- function(df, k = 3, x = NULL, y = NULL,
                             seed = NULL, max_iter = 100, tol = 1e-6) {
  if (!is.null(seed)) set.seed(seed)

  if (is.null(x) || is.null(y)) {
    num_cols <- names(df)[sapply(df, is.numeric)]
    if (length(num_cols) < 2) stop("Need at least two numeric columns for clustering.")
    if (is.null(x)) x <- num_cols[1]
    if (is.null(y)) y <- num_cols[2]
    message("Using numeric columns: x = ", x, ", y = ", y)
  }

  df <- df %>% select(all_of(c(x, y))) %>% rename(x = !!sym(x), y = !!sym(y))
  names_clusters <- paste0("C", 0:(k-1))
  df <- df %>% mutate(cluster = sample(names_clusters, n(), replace = TRUE))

  compute_centers <- function(d) {
    d %>%
      group_by(cluster) %>%
      summarise(x = mean(x), y = mean(y), .groups = "drop") %>%
      arrange(cluster)
  }

  centers <- compute_centers(df)
  iter <- 0L

  repeat {
    iter <- iter + 1L

    df_new <- df %>%
      rowwise() %>%
      mutate(cluster = centers$cluster[
        which.min(sqrt((x - centers$x)^2 + (y - centers$y)^2))
      ]) %>%
      ungroup()

    new_centers <- compute_centers(df_new)

    oc <- centers %>% arrange(cluster) %>% select(x, y) %>% as.matrix()
    nc <- new_centers %>% arrange(cluster) %>% select(x, y) %>% as.matrix()
    moved <- max(sqrt(rowSums((oc - nc)^2))) > tol

    df <- df_new
    centers <- new_centers
    if (!moved || iter >= max_iter) break
  }

  list(labels = df$cluster,
       means = centers,
       n_iter = iter,
       converged = !moved)
}


## Question 2
This is when we'll test your `kmeans` function.
#### a) Read in the `voltages_df.csv` data set. 

In [12]:
library(tidyverse)

voltages_df <- read_csv("voltages_df.csv")


Rows: 900 Columns: 250
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (250): 0, 1.00401606425703, 2.00803212851406, 3.01204819277108, 4.016064...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


#### b) Call your `kmeans` function with 3 clusters. Print the results with `results$labels` and `results$means`. 

In [13]:
library(tidyverse)
voltages_df <- read_csv("voltages_df.csv")
numeric_df <- voltages_df %>% select(where(is.numeric))
results <- kmeans_tidy(numeric_df, k = 3, seed = 42)

cat("First 10 cluster labels:\n")
print(head(results$labels, 10))

cat("\nCluster means (centroids):\n")
print(results$means)



Rows: 900 Columns: 250
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (250): 0, 1.00401606425703, 2.00803212851406, 3.01204819277108, 4.016064...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Using numeric columns: x = 0, y = 1.00401606425703



First 10 cluster labels:
 [1] "C2" "C0" "C0" "C2" "C0" "C0" "C0" "C0" "C0" "C2"

Cluster means (centroids):
# A tibble: 3 × 3
  cluster     x      y
  <chr>   <dbl>  <dbl>
1 C0      -1.03  1.28 
2 C1      -1.03 -0.576
3 C2      -1.03  1.11 


#### c) Call R's `kmeans` function with 3 clusters. Print the results with `results$labels` and `results$cluster`. 
*Hint*: Use the `as.matrix()` function to make the `voltages_df` data frame a matrix before calling `kmeans()`.

In [9]:
numeric_df <- voltages_df %>% select(where(is.numeric))

numeric_matrix <- as.matrix(numeric_df)

set.seed(42) 
r_results <- kmeans(numeric_matrix, centers = 3, nstart = 10)

cat("results$cluster (equivalent to results$labels):\n")
print(head(r_results$cluster, 10))

cat("\nCluster centroids (results$centers):\n")
print(r_results$centers)


results$cluster (equivalent to results$labels):
 [1] 1 2 2 2 2 2 3 3 2 1

Cluster centroids (results$centers):
          0 1.00401606425703 2.00803212851406 3.01204819277108 4.01606425702811
1 -1.031463        0.9381238        0.7619864        0.3631543       -1.1179412
2 -1.031463        1.2439759        1.0924697        0.9004440        0.3011754
3 -1.031463        1.3093239        1.1616772        0.9787498        0.6481497
  5.02008032128514 6.02409638554217 7.0281124497992 8.03212851405623
1        -1.051145       -0.9766807      -0.8694758       -0.6892375
2        -1.159714       -1.1098127      -1.0685484       -1.0338649
3        -1.168610       -1.1196122      -1.0590962       -0.9943176
  9.03614457831325 10.0401606425703 11.0441767068273 12.0481927710843
1       -0.5661321       -0.2497152        0.6027358        0.6960355
2       -1.0022396       -0.9699741       -0.9343697       -0.8943605
3       -0.9237437       -0.8457536       -0.7572129       -0.6201221
  13.05220883

#### d) Are your labels/clusters the same? If not, why? Are your means the same?

In [14]:

numeric_df <- voltages_df %>% select(where(is.numeric))

results <- kmeans_tidy(numeric_df, k = 3, seed = 42)

r_results <- kmeans(as.matrix(numeric_df), centers = 3, nstart = 10)

head(results$labels, 10)
head(r_results$cluster, 10)

results$means
r_results$centers


Using numeric columns: x = 0, y = 1.00401606425703



[1] "C2" "C0" "C0" "C2" "C0" "C0" "C0" "C0" "C0" "C2"

[1] 1 3 3 3 3 3 2 2 3 1

cluster,x,y
<chr>,<dbl>,<dbl>
C0,-1.031463,1.2779022
C1,-1.031463,-0.5759587
C2,-1.031463,1.1085890


,0,1.00401606425703,2.00803212851406,3.01204819277108,4.01606425702811,5.02008032128514,6.02409638554217,7.0281124497992,8.03212851405623,9.03614457831325,⋯,240.963855421687,241.967871485944,242.971887550201,243.975903614458,244.979919678715,245.983935742972,246.987951807229,247.991967871486,248.995983935743,250
1,-1.031463,0.9381238,0.7619864,0.3631543,-1.1179412,-1.051145,-0.9766807,-0.8694758,-0.6892375,-0.5661321,⋯,-0.7900387,-0.8070676,-0.8182598,-0.8207339,-0.8132928,-0.7969549,-0.77567272,-0.75689256,-0.7496483,-0.7570393
2,-1.031463,1.3093239,1.1616772,0.9787498,0.6481497,-1.168610,-1.1196122,-1.0590962,-0.9943176,-0.9237437,⋯,0.3364266,0.8337474,0.7125412,-0.2659209,-1.0409179,-1.0587745,-1.01359887,-0.96467777,-0.9151047,-0.8610245
3,-1.031463,1.2439759,1.0924697,0.9004440,0.3011754,-1.159714,-1.1098127,-1.0685484,-1.0338649,-1.0022396,⋯,-0.9107472,-0.8732292,-0.8234477,-0.7607812,-0.6682618,-0.3380864,-0.04693168,0.02820486,-0.4113500,-0.8115784


In [15]:
The cluster labels are not same because my costum function works in 2D space while R's kmeans works in 250D space.
The ffirst mean value are similar but the overall cluster centers differ as they are computed using different sets of features.

ERROR: Error in parse(text = x, srcfile = src): <text>:1:5: unexpected symbol
1: The cluster
        ^


## Question 3
#### a) Explain the process of using a for loop to assign clusters for kmeans.

In [ ]:
We loop over each observation, compute its distance to every centroid and set its cluster to the index of the closest centroid.

#### b) Explain the process of vectorizing the code to assign clusters for kmeans.

In [ ]:
We compute n*k matrix of squared distance.

#### c) State which (for loops or vectorizing) is more efficient and why.

In [ ]:
Vectorizing is more efficient. For loop goes through one point at a time and takes longer time. Vectorizing lets computer do all the maths at once for every point.

## Question 4
#### When does `kmeans` fail? What assumption does `kmeans` use that causes it to fail in this situation?

In [ ]:
K means fails when clusters are not round or when there are outliers or overlapping groups.
It assumes that all clusters are round, equal sized and the points are close to each other. If these assumptions are not true,
K-means group the points incorrectly. 

## Question 5
#### What assumption do Guassian mixture models make?

In [ ]:
It assumes that data are generated from mix of several normal distributions.
That each clusters are bell curve with its own mean, spread and orientation.

## Question 6
#### What assumption does spectral clustering make? Why does this help us?

In [ ]:
It assumes that points in same clusters are with higher similarities and weakly connected to points in other clusters.
This helps because it can find clusters that are nor round or evenly sized and detect complex shapes and patterns that k-means miss.

## Question 7
#### Define the gap statistic method. What do we use it for?

In [ ]:
Gap statistic compares how well data are clustered for different numbers of clusters to what is expected if data has no real clusters. We use it to choose best number of clusters, k.